# Imports

In [73]:
import numpy as np
import matplotlib as plt
import pandas as pd
import json

from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import ndcg_score

from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k, recall_at_k

import lightgbm as lgb


# Download Data

In [74]:
# users: user id | age | gender | occupation | zip code, tab separated
users = pd.read_csv('/Users/gracefujinaga/MSDS_490/Data/movie_lens/ml-100k/u.user', delimiter='|', dtype=None,encoding='latin-1', header=None)
columns_renamed = ['user id', 'age', 'gender', 'occupation', 'zip code']
users.columns = columns_renamed

# all data
data = pd.read_csv('/Users/gracefujinaga/MSDS_490/Data/movie_lens/ml-100k/u.data',  sep='\t', header=None)
columns_renamed = ['user_id', 'item_id', 'rating', 'timestamp']
data.columns = columns_renamed

# items
items = pd.read_csv('/Users/gracefujinaga/MSDS_490/Data/movie_lens/ml-100k/u.item', delimiter='|', dtype=None,encoding='latin-1', header=None)
columns_renamed = [
    "movie_id", "movie_title", "release_date", "video_release_date", "IMDb_URL",
    "unknown", "action", "adventure", "animation", "childrens", "comedy", "crime",
    "documentary", "drama", "fantasy", "film_noir", "horror", "musical", "mystery",
    "romance", "sci_fi", "thriller", "war", "western"
]
items.columns = columns_renamed

### Users

In [75]:
users

,user id,age,gender,occupation,zip code
0,1,24,M,technician,85711
1,2,53,F,other,94043
2,3,23,M,writer,32067
3,4,24,M,technician,43537
4,5,33,F,other,15213
...,...,...,...,...,...
938,939,26,F,student,33319
939,940,32,M,administrator,02215
940,941,20,M,student,97229
941,942,48,F,librarian,78209


### Items

In [76]:
items

,movie_id,movie_title,release_date,video_release_date,IMDb_URL,unknown,action,adventure,animation,childrens,...,fantasy,film_noir,horror,musical,mystery,romance,sci_fi,thriller,war,western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1677,1678,Mat' i syn (1997),06-Feb-1998,NaN,http://us.imdb.com/M/title-exact?Mat%27+i+syn+...,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1678,1679,B. Monkey (1998),06-Feb-1998,NaN,http://us.imdb.com/M/title-exact?B%2E+Monkey+(...,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0
1679,1680,Sliding Doors (1998),01-Jan-1998,NaN,http://us.imdb.com/Title?Sliding+Doors+(1998),0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1680,1681,You So Crazy (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?You%20So%20Cr...,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [77]:
items.drop(columns=['video_release_date', 'IMDb_URL'], inplace=True)
items = items[items['movie_id'] != 267]

In [78]:
items

,movie_id,movie_title,release_date,unknown,action,adventure,animation,childrens,comedy,crime,...,fantasy,film_noir,horror,musical,mystery,romance,sci_fi,thriller,war,western
0,1,Toy Story (1995),01-Jan-1995,0,0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1677,1678,Mat' i syn (1997),06-Feb-1998,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1678,1679,B. Monkey (1998),06-Feb-1998,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0
1679,1680,Sliding Doors (1998),01-Jan-1998,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1680,1681,You So Crazy (1994),01-Jan-1994,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


### Data

In [79]:
data

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


## Cleaning

- rescaling: x-min/(max - min)

features to use: age, gender, occupation, release_date, genre

In [80]:
# standardize age
scaler = StandardScaler()
users['age'] = scaler.fit_transform(users[['age']])

# encode gender, occupation
le_gender = LabelEncoder()
users['gender'] = le_gender.fit_transform(users['gender'])

le_occupation = LabelEncoder()
users['occupation'] = le_occupation.fit_transform(users['occupation'])


In [81]:
# clean year
items['release_date'] = pd.to_datetime(items['release_date'], errors='coerce')
items['release_year'] = items['release_date'].dt.year

# then normalize
items['release_year_scaled'] = scaler.fit_transform(items[['release_year']])

/var/folders/zv/4f9cw9vs6tjbvz5bh2k07w_40000gn/T/ipykernel_60491/1074043882.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  items['release_date'] = pd.to_datetime(items['release_date'], errors='coerce')
/var/folders/zv/4f9cw9vs6tjbvz5bh2k07w_40000gn/T/ipykernel_60491/1074043882.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  items['release_year'] = items['release_date'].dt.year
/var/folders/zv/4f9cw9vs6tjbvz5bh2k07w_40000gn/T/ipykernel_60491/1074043882.py:6: SettingWithCopyWarning: 
A value is tryin

In [82]:
# normalize ratings
data['rating_normalized'] = data['rating'] - data.groupby('user_id')['rating'].transform('mean')

In [83]:
data.rename(columns={'item_id' : 'movie_id'}, inplace=True)

In [84]:
# drop extra columns
data.drop(columns=['timestamp'], inplace=True)
items.drop(columns=['release_date', 'release_year'], inplace=True)
users.drop(columns=['zip code'], inplace=True)

/var/folders/zv/4f9cw9vs6tjbvz5bh2k07w_40000gn/T/ipykernel_60491/3326634805.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  items.drop(columns=['release_date', 'release_year'], inplace=True)


In [85]:
users.rename(columns={'user id' : 'user_id'}, inplace=True)

In [86]:
# merge the dataframes
merged_df = items.merge(data, on='movie_id').merge(users, on=['user_id'])

In [87]:
merged_df.columns

Index(['movie_id', 'movie_title', 'unknown', 'action', 'adventure',
       'animation', 'childrens', 'comedy', 'crime', 'documentary', 'drama',
       'fantasy', 'film_noir', 'horror', 'musical', 'mystery', 'romance',
       'sci_fi', 'thriller', 'war', 'western', 'release_year_scaled',
       'user_id', 'rating', 'rating_normalized', 'age', 'gender',
       'occupation'],
      dtype='object')

# Models

## Pointwise LTR (Ridge Regression)

In [213]:
genre_list = [
    'unknown', 'action', 'adventure', 'animation', 'childrens',
       'comedy', 'crime', 'documentary', 'drama', 'fantasy', 'film_noir',
       'horror', 'musical', 'mystery', 'romance', 'sci_fi', 'thriller', 'war',
       'western',
]

# set up features and test train split
features = ["age", "gender", "occupation"] + genre_list
X = merged_df[features]
y = merged_df["rating_normalized"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# fit model
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

# predict
y_pred = model.predict(X_test)

rmse_point = root_mean_squared_error(y_test, y_pred)

# evaluate
print("RMSE:", rmse_point)

RMSE: 1.020371866272053


## Pairwise model with BPR

In [214]:
# define  what a user liked
merged_df['positive'] = merged_df['rating'] >= 4  

# create dataste of user, item pairs
dataset = Dataset()
users = [x for x in merged_df['user_id']]
items = [x for x in merged_df['movie_id']]
dataset.fit(users, items)

# build interactions matrix
(interactions, weights) = dataset.build_interactions(((row['user_id'], row['movie_id']) for idx, row in merged_df[merged_df['positive']].iterrows()))

# build model
model = LightFM(loss='bpr', no_components=32, learning_rate=0.05)
model.fit(interactions, epochs=30, num_threads=4)

In [215]:
# evaluate model
k=10

precision_pair = precision_at_k(model, interactions, k=k).mean()
print(f'precision@{k}: {precision_pair:.4f}')

recall_pair = recall_at_k(model, interactions, k=k).mean()
print(f'recall@{k}: {recall_pair:.4f}')

precision@10: 0.6772
recall@10: 0.1964


In [ ]:
# ncdg 
# note that the model 0 indexes, but the movies are NOT 0 indexed, we are using the model indexes here
user_ids = np.arange(interactions.shape[0])
item_ids = np.arange(interactions.shape[1])

true_relevance = []
predicted_scores = []

for user_id in user_ids:
    # flatten to correct dimension
    true_row = interactions.tocsr()[user_id].toarray().flatten()

    # predict for each item, user pair
    scores = model.predict(np.full_like(item_ids, user_id), item_ids)

    true_relevance.append(true_row)
    predicted_scores.append(scores)

# convert to arrays
true_relevance = np.array(true_relevance)
predicted_scores = np.array(predicted_scores)

# get ncdg
nDCG_at_10_pair = ndcg_score(true_relevance, predicted_scores, k=10)
print(f"nDCG@10: {nDCG_at_10_pair:.4f}")


nDCG@10: 0.7220


## Listwise LTR 

In [222]:
# split the train and test by user
unique_users = merged_df['user_id'].unique()
train_users, test_users = train_test_split(unique_users, test_size=0.2, random_state=42)

# split the data with users
train_df = merged_df[merged_df['user_id'].isin(train_users)]
test_df = merged_df[merged_df['user_id'].isin(test_users)]

# define features and split 
genre_list = [
    'unknown', 'action', 'adventure', 'animation', 'childrens',
       'comedy', 'crime', 'documentary', 'drama', 'fantasy', 'film_noir',
       'horror', 'musical', 'mystery', 'romance', 'sci_fi', 'thriller', 'war',
       'western',
]
features = ["age", "gender", "occupation"] + genre_list

X_train = train_df[features]  
y_train = train_df['rating']

X_test = test_df[features]
y_test = test_df['rating']

# items per user count
group_train = train_df.groupby('user_id').size().values
group_test = test_df.groupby('user_id').size().values

# set up test and train db
lgb_train = lgb.Dataset(X_train, y_train, group=group_train)
lgb_test = lgb.Dataset(X_test, y_test, group=group_test, reference=lgb_train)

# run model with params
params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'learning_rate': 0.05,
}

model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_test],
    valid_names=['train', 'valid']
)

In [223]:
# predict
predicted_scores = model.predict(X_test)
test_df = test_df.copy()
test_df['predicted_score'] = predicted_scores

# sort by ranking
ranked_df = test_df.sort_values(['user_id', 'predicted_score'], ascending=[True, False])

In [ ]:
# remove unnecessary columns
ranked_df.drop(columns = genre_list + ['age', 'gender', 'occupation', 'positive' ], inplace=True)

In [201]:
# get top 10 movies for the user
def get_top_k(user_id, k):
    top_k = ranked_df[ranked_df['user_id'] == user_id][:k]
    return top_k

get_top_k(921, 10)

,movie_id,movie_title,release_year_scaled,user_id,rating,rating_normalized,predicted_score
45491,50,Star Wars (1977),-0.869239,921,4,0.727273,1.516577
45515,181,Return of the Jedi (1983),0.534335,921,5,1.727273,1.516577
45512,172,"Empire Strikes Back, The (1980)",-0.658703,921,4,0.727273,1.508563
45543,313,Titanic (1997),0.534335,921,5,1.727273,1.238653
45517,190,Henry V (1989),-0.027095,921,2,-1.272727,0.418181
45558,471,Courage Under Fire (1996),0.464156,921,2,-1.272727,0.418181
45507,133,Gone with the Wind (1939),-3.536029,921,5,1.727273,0.345667
45565,651,Glory (1989),-0.027095,921,3,-0.272727,0.295031
45560,484,"Maltese Falcon, The (1941)",-3.395672,921,3,-0.272727,0.246104
45493,69,Forrest Gump (1994),0.323799,921,4,0.727273,0.238294


In [ ]:
k = 10

# set up
y_true, y_pred = [], []
precision_list = []
recall_list = []

# loop through each user
for user_id in ranked_df['user_id'].unique():
    user_ranked_df = ranked_df[ranked_df['user_id'] == user_id]
    true_relevance = user_ranked_df['rating'].values
    predicted = user_ranked_df['predicted_score'].values

    # actual
    relevant_mask = true_relevance >= 4.0
    num_relevant = np.sum(relevant_mask)

    # predicted
    top_k_idx = np.argsort(predicted)[::-1][:k]
    top_k_relevant = relevant_mask[top_k_idx]

    precision = np.sum(top_k_relevant) / k
    recall = np.sum(top_k_relevant) / num_relevant if num_relevant > 0 else 0

    precision_list.append(precision)
    recall_list.append(recall)

    # For nDCG
    y_true.append(true_relevance[:k])
    y_pred.append(predicted[:k])

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# compute
ncdg_list = ndcg_score(y_true, y_pred, k=k)
precision_list = np.mean(precision_list)
recall_list = np.mean(recall_list)

print(f"nDCG@{k}: {ncdg_list:.4f}")
print(f"Precision@{k}: {precision_list:.4f}")
print(f"Recall@{k}: {recall_list:.4f}")


nDCG@10: 0.9422
Precision@10: 0.7212
Recall@10: 0.2279


# Compare across models

In [221]:
metrics = ['model', 'RMSE', 'Precision@10', 'Recall@10', 'NCDG@10']
results = [['pointwise', rmse_point, None, None, None],
           ['pairwise', None, precision_pair, recall_pair, nDCG_at_10_pair],
           ['listwise', None, precision_list, recall_list, ncdg_list]]

results_df = pd.DataFrame(results)
results_df.columns = metrics
results_df

,model,RMSE,Precision@10,Recall@10,NCDG@10
0,pointwise,1.020372,NaN,NaN,NaN
1,pairwise,NaN,0.677176,0.196389,0.721998
2,listwise,NaN,0.721164,0.227939,0.942185
